In [1]:
import os
import cv2
import shutil
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO

# ================= CONFIGURATION =================
ROOT = "get_project_root()"
BDD_PATH = os.path.join(ROOT, "data/03_inprocess/BDD")

# Poids YOLO (Détection + Classification basique si le modèle a été entrainé sur 2 classes)
YOLO_WEIGHTS = os.path.join(ROOT, "outputs/yolov11n_i1024b2_e100/det/weights/best.pt")

# Mapping des dossiers source vers destination
# Structure BDD source : {NOM_DOSSIER} -> images / labels
# Structure Cible : {OUTPUT_TYPE} -> male / femelle
DIRS_MAPPING = {
    "Nouvelles": "tube",
    "Croppées": "unblur"
}

# Paramètres
CONF_THRESHOLD = 0.25
IMG_SIZE = 1024
MARGIN = 10  # Marge en pixels autour de la boite de détection

# ================= FONCTIONS =================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def get_label_path(img_path):
    """Retrouve le fichier label associé à une image."""
    path_obj = Path(img_path)
    # Remonte d'un niveau (images) et va dans labels
    label_dir = path_obj.parent.parent / "labels"
    label_file = label_dir / (path_obj.stem + ".txt")
    return label_file

def save_crop(img, box, out_dir, base_name, idx, class_name):
    """Découpe et sauvegarde le crop."""
    h, w, _ = img.shape
    x1, y1, x2, y2 = map(int, box)
    
    # Ajout d'une marge de sécurité (sans dépasser l'image)
    x1 = max(0, x1 - MARGIN)
    y1 = max(0, y1 - MARGIN)
    x2 = min(w, x2 + MARGIN)
    y2 = min(h, y2 + MARGIN)
    
    crop = img[y1:y2, x1:x2]
    
    if crop.size == 0:
        return

    # Nettoyage du nom de classe pour le fichier
    safe_cls_name = class_name.replace(" ", "_").lower()
    
    # Création du sous-dossier de classe (male/femelle)
    # On normalise ici : si YOLO renvoie 'gammare_male', on met dans 'male'
    target_subfolder = "male" if "male" in safe_cls_name else "femelle" if "femelle" in safe_cls_name else "unknown"
    
    final_dir = os.path.join(out_dir, target_subfolder)
    ensure_dir(final_dir)
    
    out_name = f"{base_name}_c{idx}_{safe_cls_name}.jpg"
    cv2.imwrite(os.path.join(final_dir, out_name), crop)

# ================= MAIN PIPELINE =================

def run_processing():
    # 1. Chargement du modèle
    if not os.path.exists(YOLO_WEIGHTS):
        raise FileNotFoundError(f"Poids YOLO introuvables : {YOLO_WEIGHTS}")
    
    print(f"Chargement du modèle : {YOLO_WEIGHTS}")
    model = YOLO(YOLO_WEIGHTS)
    
    # Récupération des noms de classes du modèle pour le mapping
    model_names = model.names
    print(f"Classes détectées par le modèle : {model_names}")

    # 2. Parcours des dossiers
    for source_folder, target_type in DIRS_MAPPING.items():
        src_img_dir = os.path.join(BDD_PATH, source_folder, "images")
        out_root = os.path.join(ROOT, "data/04_crops_output", target_type) # Sortie organisée
        
        if not os.path.exists(src_img_dir):
            print(f"⚠️ Dossier source introuvable : {src_img_dir}, passage au suivant.")
            continue
            
        print(f"\nTraitement : {source_folder} -> {target_type}")
        print(f"Sortie vers : {out_root}")
        
        image_files = [f for f in os.listdir(src_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        for img_file in tqdm(image_files, desc=f"Processing {source_folder}"):
            img_path = os.path.join(src_img_dir, img_file)
            
            # A. Vérification Label (Critère bloquant demandé)
            label_path = get_label_path(img_path)
            if not label_path.exists():
                # print(f"Ignoré (pas de label) : {img_file}") # Decommenter pour debug
                continue
                
            # B. Chargement Image
            img = cv2.imread(img_path)
            if img is None:
                continue
                
            # C. Inférence YOLO
            results = model.predict(source=img, imgsz=IMG_SIZE, conf=CONF_THRESHOLD, verbose=False)
            result = results[0]
            
            # D. Traitement des détections
            boxes = result.boxes.xyxy.cpu().numpy()
            cls_ids = result.boxes.cls.cpu().numpy()
            
            if len(boxes) == 0:
                continue
                
            base_filename = os.path.splitext(img_file)[0]
            
            for i, box in enumerate(boxes):
                cls_id = int(cls_ids[i])
                class_name = model_names[cls_id]
                
                # Sauvegarde du crop dans le bon dossier
                save_crop(img, box, out_root, base_filename, i, class_name)

if __name__ == "__main__":
    run_processing()

Chargement du modèle : get_project_root()outputs/yolov11n_i1024b2_e100/det/weights/best.pt
Classes détectées par le modèle : {0: 'item'}

Traitement : Nouvelles -> tube
Sortie vers : get_project_root()data/04_crops_output/tube


Processing Nouvelles: 100%|███████████████████| 532/532 [00:14<00:00, 36.31it/s]



Traitement : Croppées -> unblur
Sortie vers : get_project_root()data/04_crops_output/unblur


Processing Croppées: 100%|███████████████████| 193/193 [00:00<00:00, 215.68it/s]
